In [1]:
import contextlib
from pathlib import Path
from uuid import uuid4

import polars as pl
import torch
import torch.nn.functional as F
from IPython.display import display
from lightning import Fabric

from src.chart_transport.constraint import (
    LagrangianConstraintConfig,
    LossConstraintConfig,
)
from src.common.training import TrainingConfig
from src.data.gaussian_mixture.data import MultimodalGaussianDataConfig
from src.experiments.multimodal_gaussian.canonical import (
    get_canonical_chart_transport_configs,
)

torch.set_float32_matmul_precision("medium")


In [2]:
num_modes = 4
mode_std = 0.1
offset = 1.0
ambient_dimension = 8

pretrain_chart_n_steps = 1_000
pretrain_critic_n_steps = 1_000
update_chart_every_n_critic_steps = 5
integrated_n_steps = 4_000
seed = 0
log_every_n_steps = 100
boundary_eps = 1e-3

data_config = MultimodalGaussianDataConfig.initialize(
    num_modes=num_modes,
    mode_std=mode_std,
    offset=offset,
    ambient_dimension=ambient_dimension,
)

chart_transport_config = get_canonical_chart_transport_configs(
    data_config=data_config,
)

training_config = TrainingConfig.initialize(
    train_batch_size=2048,
    eval_batch_size=1024,
    eval_every_n_training_steps=3000,
    folder=(
        Path("/home/nlyu/Code/diffusive-latent-generation/artifacts/multimodal_gaussian")
        / uuid4().hex
    ),
)

chart_transport_config.visualize()


In [ ]:
fabric = Fabric(
    accelerator="cuda", devices=1,
    precision="bf16-mixed",
)
fabric.launch()
fabric.seed_everything(seed)

device = fabric.device
device_type = device.type

runtime_data_config = data_config.replace(
    path="isometry",
    replacement=data_config.isometry.to(device=device, dtype=torch.float32),
).replace(
    path="projection",
    replacement=data_config.projection.to(device=device, dtype=torch.float32),
)

architecture_config = chart_transport_config.architecture_config
chart_transport_model = architecture_config.get_model()
optimizer = architecture_config.get_optimizer(
    model=chart_transport_model,
)
chart_transport_model, optimizer = fabric.setup(chart_transport_model, optimizer)

encoder = chart_transport_model.encoder
decoder = chart_transport_model.decoder
critic = chart_transport_model.critic

prior_config = chart_transport_config.prior_config
constraint_config = chart_transport_config.loss_config.constraint_config
constraint_method = constraint_config.constraint_method
pretrain_config = chart_transport_config.loss_config.chart_pretrain_config
transport_config = chart_transport_config.loss_config.transport_config

data_dual = torch.zeros((), device=device)
prior_dual = torch.zeros((), device=device)

fabric.print(
    f"device={device}, precision=bf16-mixed, folder={training_config.folder}"
)

Using bfloat16 Automatic Mixed Precision (AMP)
Seed set to 0


device=cuda:0, precision=bf16-mixed, folder=/home/nlyu/Code/diffusive-latent-generation/artifacts/multimodal_gaussian/b0dcc174fcf44afdabf9d424f2027acd


In [14]:
class PretrainMonitorConfig(BaseConfig):
    log_every_n_steps: int
    plot_n_sample_pairs: int # Initialize to 1000
    plot_n_data_latents_per_mode: int # Initialize to 1000

class MonitorConfig(BaseConfig):
    pretrain_monitor_config: PretrainMonitorConfig

NameError: name 'BaseConfig' is not defined

## Pretrain

In [4]:
pretrain_history = []
encoder.train()
decoder.train()
critic.eval()

for step in range(1, chart_transport_config.scheduling_config.pretrain_chart_n_steps + 1):
    optimizer.zero_grad(set_to_none=True)

    with contextlib.ExitStack() as stack:
        stack.enter_context(torch.cuda.device(device))
        stack.enter_context(torch.device(str(device)))
        stack.enter_context(torch.autocast(device_type=device_type, dtype=torch.bfloat16))

        data_batch = runtime_data_config.sample_unconditional(
            batch_size=training_config.train_batch_size,
        )
        prior_batch = prior_config.sample(
            batch_size=training_config.train_batch_size,
        ).to(device=device, dtype=torch.float32)

        data_latents = encoder(data_batch)
        data_reconstruction = decoder(data_latents)
        prior_reconstruction = decoder(prior_batch)
        prior_latents = encoder(prior_reconstruction)

        data_cycle_loss = F.huber_loss(
            data_reconstruction,
            data_batch,
            delta=constraint_config.huber_delta,
            reduction="mean",
        )
        prior_cycle_loss = F.huber_loss(
            prior_latents,
            prior_batch,
            delta=constraint_config.huber_delta,
            reduction="mean",
        )
        constraint_loss = data_cycle_loss + prior_cycle_loss

        zero_mean_loss = data_latents.mean(dim=0).square().mean()
        latent_norms = data_latents.square().sum(dim=-1).sqrt()
        softplus_loss = F.softplus(
            latent_norms - pretrain_config.softplus_radius,
        ).mean()
        chart_loss = constraint_loss
        chart_loss = chart_loss + pretrain_config.zero_mean_weight * zero_mean_loss
        chart_loss = chart_loss + pretrain_config.softplus_weight * softplus_loss

    fabric.backward(chart_loss)
    fabric.clip_gradients(
        chart_transport_model,
        optimizer,
        max_norm=architecture_config.grad_clip_norm,
    )
    optimizer.step()

    metrics = {
        "stage": "pretrain",
        "step": step,
        "chart_loss": chart_loss.detach().item(),
        "data_cycle_loss": data_cycle_loss.detach().item(),
        "prior_cycle_loss": prior_cycle_loss.detach().item(),
        "zero_mean_loss": zero_mean_loss.detach().item(),
        "softplus_loss": softplus_loss.detach().item(),
    }
    pretrain_history.append(metrics)

    if step == 1 or step % log_every_n_steps == 0 or step == chart_transport_config.scheduling_config.pretrain_chart_n_steps:
        summary = ", ".join(
            f"{key}={value:.4f}"
            for key, value in metrics.items()
            if isinstance(value, float)
        )
        fabric.print(f"[pretrain] step {step}/{chart_transport_config.scheduling_config.pretrain_chart_n_steps}: {summary}")

pretrain_history = pl.DataFrame(pretrain_history)
display(pretrain_history.tail())


[pretrain] step 1/1000: chart_loss=0.7013, data_cycle_loss=0.1729, prior_cycle_loss=0.5276, zero_mean_loss=0.0559, softplus_loss=0.0162
[pretrain] step 100/1000: chart_loss=0.0024, data_cycle_loss=0.0001, prior_cycle_loss=0.0012, zero_mean_loss=0.0882, softplus_loss=0.0204
[pretrain] step 200/1000: chart_loss=0.0080, data_cycle_loss=0.0022, prior_cycle_loss=0.0050, zero_mean_loss=0.0589, softplus_loss=0.0183
[pretrain] step 300/1000: chart_loss=0.0016, data_cycle_loss=0.0004, prior_cycle_loss=0.0004, zero_mean_loss=0.0584, softplus_loss=0.0182
[pretrain] step 400/1000: chart_loss=0.0075, data_cycle_loss=0.0029, prior_cycle_loss=0.0029, zero_mean_loss=0.1401, softplus_loss=0.0230
[pretrain] step 500/1000: chart_loss=0.0010, data_cycle_loss=0.0001, prior_cycle_loss=0.0002, zero_mean_loss=0.0492, softplus_loss=0.0185
[pretrain] step 600/1000: chart_loss=0.0008, data_cycle_loss=0.0000, prior_cycle_loss=0.0002, zero_mean_loss=0.0400, softplus_loss=0.0175
[pretrain] step 700/1000: chart_loss

stage,step,chart_loss,data_cycle_loss,prior_cycle_loss,zero_mean_loss,softplus_loss
str,i64,f64,f64,f64,f64,f64
"""pretrain""",996,0.000233,0.00002,0.000058,0.000387,0.015076
"""pretrain""",997,0.000259,0.000022,0.000083,0.000288,0.015074
"""pretrain""",998,0.000279,0.000022,0.000104,0.000257,0.015064
"""pretrain""",999,0.000261,0.000021,0.000086,0.000371,0.015063
"""pretrain""",1000,0.00036,0.000021,0.000184,0.000411,0.015048


## Visualize noise

In [5]:
import hvplot.polars

pretrain_history.hvplot.line(
    x="step",
    y=["chart_loss", "data_cycle_loss"]
)

:NdOverlay   [Variable]
   :Curve   [step]   (value)

## Train noise critic

In [6]:
critic_pretrain_history = []
encoder.eval()
decoder.eval()
critic.train()

for step in range(1, chart_transport_config.scheduling_config.pretrain_critic_n_steps + 1):
    optimizer.zero_grad(set_to_none=True)

    with contextlib.ExitStack() as stack:
        if device_type == "cuda":
            stack.enter_context(torch.cuda.device(device))
        stack.enter_context(torch.device(str(device)))
        stack.enter_context(torch.autocast(device_type=device_type, dtype=torch.bfloat16))

        data_batch = runtime_data_config.sample_unconditional(
            batch_size=training_config.train_batch_size,
        )
        with torch.no_grad():
            data_latents = encoder(data_batch)

        t = boundary_eps + (1.0 - 2.0 * boundary_eps) * torch.rand(
            data_latents.shape[0],
            device=device,
        )
        eps = torch.randn_like(data_latents)
        noised_latents = (
            (1.0 - t).unsqueeze(-1) * data_latents + t.unsqueeze(-1) * eps
        )
        predicted_noise = critic(noised_latents, t)
        critic_loss = F.mse_loss(predicted_noise, eps)

    fabric.backward(critic_loss)
    fabric.clip_gradients(
        chart_transport_model,
        optimizer,
        max_norm=architecture_config.grad_clip_norm,
    )
    optimizer.step()

    metrics = {
        "stage": "critic_pretrain",
        "step": step,
        "critic_loss": critic_loss.detach().item(),
    }
    critic_pretrain_history.append(metrics)

    if step == 1 or step % log_every_n_steps == 0 or step == chart_transport_config.scheduling_config.pretrain_critic_n_steps:
        summary = ", ".join(
            f"{key}={value:.4f}"
            for key, value in metrics.items()
            if isinstance(value, float)
        )
        fabric.print(f"[critic_pretrain] step {step}/{chart_transport_config.scheduling_config.pretrain_critic_n_steps}: {summary}")

critic_pretrain_history = pl.DataFrame(critic_pretrain_history)
display(critic_pretrain_history.tail())


[critic_pretrain] step 1/2000: critic_loss=0.8790
[critic_pretrain] step 100/2000: critic_loss=0.2049
[critic_pretrain] step 200/2000: critic_loss=0.1711
[critic_pretrain] step 300/2000: critic_loss=0.1711
[critic_pretrain] step 400/2000: critic_loss=0.1700
[critic_pretrain] step 500/2000: critic_loss=0.1688
[critic_pretrain] step 600/2000: critic_loss=0.1591
[critic_pretrain] step 700/2000: critic_loss=0.1557
[critic_pretrain] step 800/2000: critic_loss=0.1501
[critic_pretrain] step 900/2000: critic_loss=0.1443
[critic_pretrain] step 1000/2000: critic_loss=0.1714
[critic_pretrain] step 1100/2000: critic_loss=0.1563
[critic_pretrain] step 1200/2000: critic_loss=0.1573
[critic_pretrain] step 1300/2000: critic_loss=0.1379
[critic_pretrain] step 1400/2000: critic_loss=0.1394
[critic_pretrain] step 1500/2000: critic_loss=0.1637
[critic_pretrain] step 1600/2000: critic_loss=0.1419
[critic_pretrain] step 1700/2000: critic_loss=0.1488
[critic_pretrain] step 1800/2000: critic_loss=0.1586
[crit

stage,step,critic_loss
str,i64,f64
"""critic_pretrain""",1996,0.133364
"""critic_pretrain""",1997,0.135803
"""critic_pretrain""",1998,0.156194
"""critic_pretrain""",1999,0.124041
"""critic_pretrain""",2000,0.16208


## Integrated training

In [7]:
integrated_history = []
critic_steps_per_chart_step = max(
    1,
    chart_transport_config.scheduling_config.update_chart_every_n_critic_steps,
)

for step in range(1, integrated_n_steps + 1):
    encoder.eval()
    decoder.eval()
    critic.train()
    optimizer.zero_grad(set_to_none=True)

    with contextlib.ExitStack() as stack:
        if device_type == "cuda":
            stack.enter_context(torch.cuda.device(device))
        stack.enter_context(torch.device(str(device)))
        stack.enter_context(torch.autocast(device_type=device_type, dtype=torch.bfloat16))

        critic_data_batch = runtime_data_config.sample_unconditional(
            batch_size=training_config.train_batch_size,
        )
        with torch.no_grad():
            critic_data_latents = encoder(critic_data_batch)

        critic_t = boundary_eps + (1.0 - 2.0 * boundary_eps) * torch.rand(
            critic_data_latents.shape[0],
            device=device,
        )
        critic_eps = torch.randn_like(critic_data_latents)
        critic_noised_latents = (
            (1.0 - critic_t).unsqueeze(-1) * critic_data_latents
            + critic_t.unsqueeze(-1) * critic_eps
        )
        critic_predicted_noise = critic(critic_noised_latents, critic_t)
        critic_loss = F.mse_loss(critic_predicted_noise, critic_eps)

    fabric.backward(critic_loss)
    fabric.clip_gradients(
        chart_transport_model,
        optimizer,
        max_norm=architecture_config.grad_clip_norm,
    )
    optimizer.step()

    metrics = {
        "stage": "integrated",
        "step": step,
        "critic_loss": critic_loss.detach().item(),
        "chart_loss": float("nan"),
        "encoder_transport_loss": float("nan"),
        "decoder_transport_loss": float("nan"),
        "transport_field_norm": float("nan"),
        "data_cycle_loss": float("nan"),
        "prior_cycle_loss": float("nan"),
        "data_dual": data_dual.detach().item(),
        "prior_dual": prior_dual.detach().item(),
    }

    if step % critic_steps_per_chart_step == 0:
        encoder.train()
        decoder.train()
        optimizer.zero_grad(set_to_none=True)

        with contextlib.ExitStack() as stack:
            if device_type == "cuda":
                stack.enter_context(torch.cuda.device(device))
            stack.enter_context(torch.device(str(device)))
            stack.enter_context(torch.autocast(device_type=device_type, dtype=torch.bfloat16))

            chart_data_batch = runtime_data_config.sample_unconditional(
                batch_size=training_config.train_batch_size,
            )
            chart_prior_batch = prior_config.sample(
                batch_size=training_config.train_batch_size,
            ).to(device=device, dtype=torch.float32)

            chart_data_latents = encoder(chart_data_batch)
            chart_data_reconstruction = decoder(chart_data_latents)
            chart_prior_reconstruction = decoder(chart_prior_batch)
            chart_prior_latents = encoder(chart_prior_reconstruction)

            data_cycle_loss = F.huber_loss(
                chart_data_reconstruction,
                chart_data_batch,
                delta=constraint_config.huber_delta,
                reduction="mean",
            )
            prior_cycle_loss = F.huber_loss(
                chart_prior_latents,
                chart_prior_batch,
                delta=constraint_config.huber_delta,
                reduction="mean",
            )

            if isinstance(constraint_method, LossConstraintConfig):
                constraint_loss = (
                    constraint_method.data_loss_weight * data_cycle_loss
                    + constraint_method.prior_loss_weight * prior_cycle_loss
                )
            else:
                constraint_loss = data_cycle_loss + prior_cycle_loss
                constraint_loss = constraint_loss + data_dual * (
                    data_cycle_loss - constraint_method.data_constraint_budget
                )
                constraint_loss = constraint_loss + prior_dual * (
                    prior_cycle_loss - constraint_method.prior_constraint_budget
                )

            with torch.no_grad():
                transport_source_latents = encoder(chart_data_batch)
                time_bins = torch.arange(
                    transport_config.num_time_samples,
                    device=device,
                    dtype=torch.float32,
                )
                time_jitter = torch.rand(
                    transport_source_latents.shape[0],
                    transport_config.num_time_samples,
                    device=device,
                )
                transport_t = (time_bins.unsqueeze(0) + time_jitter) / transport_config.num_time_samples
                transport_t = transport_t.clamp(min=boundary_eps, max=1.0 - boundary_eps)

                transport_source_latents = transport_source_latents.unsqueeze(1).expand(
                    -1,
                    transport_config.num_time_samples,
                    -1,
                )
                transport_eps = torch.randn(
                    transport_source_latents.shape[0],
                    transport_config.num_time_samples,
                    transport_source_latents.shape[-1],
                    device=device,
                    dtype=transport_source_latents.dtype,
                )

                if transport_config.antipodal_estimate:
                    transport_t = torch.cat([transport_t, transport_t], dim=1)
                    transport_source_latents = transport_source_latents.repeat(1, 2, 1)
                    transport_eps = torch.cat([transport_eps, -transport_eps], dim=1)

                transport_noised_latents = (
                    (1.0 - transport_t).unsqueeze(-1) * transport_source_latents
                    + transport_t.unsqueeze(-1) * transport_eps
                )
                flat_transport_noised_latents = transport_noised_latents.reshape(
                    -1,
                    transport_noised_latents.shape[-1],
                )
                flat_transport_t = transport_t.reshape(-1)

                transport_predicted_noise = critic(
                    flat_transport_noised_latents,
                    flat_transport_t,
                ).reshape_as(transport_noised_latents)
                transport_prior_score = prior_config.analytic_score(
                    flat_transport_noised_latents.float(),
                    flat_transport_t.float(),
                ).to(dtype=flat_transport_noised_latents.dtype).reshape_as(
                    transport_noised_latents
                )
                transport_pullback_weight = (
                    transport_config.kl_weight_schedule.pullback_weight(
                        flat_transport_t.float(),
                    )
                    .to(dtype=flat_transport_noised_latents.dtype)
                    .reshape_as(transport_t)
                )
                transport_field_terms = transport_pullback_weight.unsqueeze(-1) * (
                    transport_prior_score
                    + transport_predicted_noise / transport_t.unsqueeze(-1)
                )

                if transport_config.antipodal_estimate:
                    midpoint = transport_config.num_time_samples
                    transport_field_terms = 0.5 * (
                        transport_field_terms[:, :midpoint]
                        + transport_field_terms[:, midpoint:]
                    )

                transport_field = transport_field_terms.mean(dim=1)
                transported_latents = (
                    encoder(chart_data_batch)
                    + transport_config.transport_step_size * transport_field
                )

            live_latents = encoder(chart_data_batch)
            encoder_transport_loss = F.huber_loss(
                live_latents,
                transported_latents,
                delta=transport_config.huber_delta,
                reduction="mean",
            )
            decoder_transport_loss = F.huber_loss(
                decoder(transported_latents),
                chart_data_batch,
                delta=transport_config.huber_delta,
                reduction="mean",
            )
            chart_loss = constraint_loss
            chart_loss = chart_loss + (
                transport_config.encoder_transport_weight * encoder_transport_loss
            )
            chart_loss = chart_loss + (
                transport_config.decoder_transport_weight * decoder_transport_loss
            )

        fabric.backward(chart_loss)
        fabric.clip_gradients(
            chart_transport_model,
            optimizer,
            max_norm=architecture_config.grad_clip_norm,
        )
        optimizer.step()

        if isinstance(constraint_method, LagrangianConstraintConfig):
            data_dual = (
                data_dual
                + constraint_method.dual_variable_lr
                * (data_cycle_loss.detach() - constraint_method.data_constraint_budget)
            ).clamp_min(0.0)
            prior_dual = (
                prior_dual
                + constraint_method.dual_variable_lr
                * (prior_cycle_loss.detach() - constraint_method.prior_constraint_budget)
            ).clamp_min(0.0)

        metrics["chart_loss"] = chart_loss.detach().item()
        metrics["encoder_transport_loss"] = encoder_transport_loss.detach().item()
        metrics["decoder_transport_loss"] = decoder_transport_loss.detach().item()
        metrics["transport_field_norm"] = transport_field.norm(dim=-1).mean().detach().item()
        metrics["data_cycle_loss"] = data_cycle_loss.detach().item()
        metrics["prior_cycle_loss"] = prior_cycle_loss.detach().item()
        metrics["data_dual"] = data_dual.detach().item()
        metrics["prior_dual"] = prior_dual.detach().item()

    integrated_history.append(metrics)

    if step == 1 or step % log_every_n_steps == 0 or step == integrated_n_steps:
        summary = ", ".join(
            f"{key}={value:.4f}"
            for key, value in metrics.items()
            if isinstance(value, float)
        )
        fabric.print(f"[integrated] step {step}/{integrated_n_steps}: {summary}")

integrated_history = pl.DataFrame(integrated_history)
display(integrated_history.tail())


[integrated] step 1/4000: critic_loss=0.1482, chart_loss=0.0828, encoder_transport_loss=0.0321, decoder_transport_loss=0.0506, transport_field_norm=2.2968, data_cycle_loss=0.0000, prior_cycle_loss=0.0001, data_dual=0.0000, prior_dual=0.0000
[integrated] step 100/4000: critic_loss=0.4584, chart_loss=0.1956, encoder_transport_loss=0.0954, decoder_transport_loss=0.0433, transport_field_norm=3.1488, data_cycle_loss=0.0255, prior_cycle_loss=0.0312, data_dual=0.0018, prior_dual=0.0068
[integrated] step 200/4000: critic_loss=0.5063, chart_loss=0.0801, encoder_transport_loss=0.0542, decoder_transport_loss=0.0136, transport_field_norm=2.6404, data_cycle_loss=0.0037, prior_cycle_loss=0.0087, data_dual=0.0021, prior_dual=0.0077
[integrated] step 300/4000: critic_loss=0.5225, chart_loss=0.5872, encoder_transport_loss=0.4873, decoder_transport_loss=0.0503, transport_field_norm=9.4376, data_cycle_loss=0.0078, prior_cycle_loss=0.0415, data_dual=0.0018, prior_dual=0.0094
[integrated] step 400/4000: cr

stage,step,critic_loss,chart_loss,encoder_transport_loss,decoder_transport_loss,transport_field_norm,data_cycle_loss,prior_cycle_loss,data_dual,prior_dual
str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""integrated""",3996,0.401014,0.030035,0.01169,0.006055,1.331418,0.003078,0.009244,0.0,0.040925
"""integrated""",3997,0.385374,0.03044,0.01543,0.005162,1.403449,0.002528,0.007426,0.0,0.040922
"""integrated""",3998,0.373665,0.026921,0.015569,0.004316,1.496528,0.002001,0.005231,0.0,0.040918
"""integrated""",3999,0.359619,0.037555,0.025956,0.005391,1.855303,0.001622,0.004798,0.0,0.040912
"""integrated""",4000,0.372034,0.042616,0.029633,0.007714,2.148375,0.001481,0.004033,0.0,0.040907


In [9]:
integrated_history.hvplot.line(
    x="step",
    y=["chart_loss", "encoder_transport_loss", "decoder_transport_loss"]
)

:NdOverlay   [Variable]
   :Curve   [step]   (value)

In [12]:
integrated_history.hvplot.line(
    x="step",
    y=["data_cycle_loss", "prior_cycle_loss"]
)

:NdOverlay   [Variable]
   :Curve   [step]   (value)

In [11]:
integrated_history.hvplot.line(
    x="step",
    y=["transport_field_norm"]
)

:Curve   [step]   (transport_field_norm)